In [2]:
import torch
torch._dynamo.disable()
import random
import time
import gymnasium as gym
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from gymnasium import Env
from torch.distributions import Categorical


def set_seed(seed: int):
    random.seed(seed) # Python
    np.random.seed(seed) # NumPy
    torch.manual_seed(seed) # PyTorch (CPU)
    torch.cuda.manual_seed(seed) # PyTorch (GPU)
    torch.cuda.manual_seed_all(seed) # PyTorch (GPU)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def make_cartpole_env(render: bool = False) -> Env:
    env_id = "CartPole-v1"
    if not render:
        env = gym.make(env_id)
        return env
    try:
        env = gym.make(env_id, render_mode="human")
    except TypeError:
        env = gym.make(env_id)
    return env

def reset_env(env: Env, seed: int | None = None):
    if seed is not None:
        try:
            state = env.reset(seed=seed)
        except TypeError:
            env.seed(seed)
            state = env.reset()
    else:
        state = env.reset()

    if isinstance(state, tuple):
        state, info = state
    return state

def step_env(env: Env, action):
    out = env.step(action)
    if len(out) == 5:
        state, reward, terminated, truncated, info = out
        done = terminated or truncated
    else:
        state, reward, done, info = out
    return state, reward, done, info



def compute_returns(rewards, gamma: float):
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return returns

class PolicyNet(nn.Module):
  def __init__(self, state_dim, action_dim, hidden_dim):
      super().__init__()
      self.fc1 = nn.Linear(state_dim, hidden_dim)
      self.fc2 = nn.Linear(hidden_dim, action_dim)
  def forward(self, state:torch.Tensor) -> torch.Tensor:
      x = F.relu(self.fc1(state))
      logits = self.fc2(x)
      return logits

def train_cartpole(gamma=0.99, num_episodes=1000, hidden_dim=128, lr=1e-2, seed=0):
    set_seed(seed)
    env = make_cartpole_env(render=False)
    state_dim = env.observation_space.shape[0] # 4
    action_dim = env.action_space.n # 2
    policy_net = PolicyNet(state_dim, action_dim, hidden_dim)
    optimizer = optim.Adam(policy_net.parameters(), lr=lr)

    episode_rewards = [] # total reward per episode
    for episode in range(1, num_episodes + 1):
        state = reset_env(env, seed + episode)
        log_probs = []
        rewards = []
        done = False
        while not done:
            action, log_prob = select_action(policy_net, state)
            state, reward, done, info = step_env(env, action)
            log_probs.append(log_prob)
            rewards.append(reward)
        returns = compute_returns(rewards, gamma)
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        policy_loss = 0
        for log_prob, G_t in zip(log_probs, returns):
            policy_loss += -log_prob * G_t
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        total_reward = sum(rewards)
        episode_rewards.append(total_reward)
        moving_avg = np.mean(episode_rewards[-20:])

        if episode % 20 == 0:
            print(f"Episode {episode:4d} | "
                f"Return: {total_reward:6.1f} | "
                f"Moving avg (last 20): {moving_avg:6.1f}")
        if moving_avg >= 475.0 and episode >= 50:
            print(f"\nEnvironment solved in {episode} episodes!")
            break
    env.close()
    return policy_net

def get_action_probs(policy_net: PolicyNet, state_np: np.ndarray):
    state_tensor = torch.from_numpy(state_np)
    logits = policy_net(state_tensor)
    probs = torch.softmax(logits, dim=-1)
    return probs

def select_action(policy_net: PolicyNet, state_np: np.ndarray):
    probs = get_action_probs(policy_net, state_np)
    m = Categorical(probs)
    action = m.sample()
    log_prob = m.log_prob(action)
    return action.item(), log_prob

def watch_agent_gui(policy_net: PolicyNet, num_episodes: int = 3):
    env = make_cartpole_env(render=True)

    for ep in range(1, num_episodes + 1):
        state = reset_env(env)
        total_reward = 0
        step = 0
        done = False

        while not done:
            time.sleep(0.02)

            with torch.no_grad():
                probs = get_action_probs(policy_net, state)
                action = torch.argmax(probs).item()

            state, reward, done, info = step_env(env, action)
            total_reward += reward
            step += 1

        print(f"[GUI] Episode {ep}: total reward = {total_reward}, steps = {step}")

    env.close()

if __name__ == "__main__":
    trained_policy = train_cartpole()
    watch_agent_gui(trained_policy, num_episodes=3)

Episode   20 | Return:   10.0 | Moving avg (last 20):   13.9
Episode   40 | Return:    8.0 | Moving avg (last 20):    9.9
Episode   60 | Return:   11.0 | Moving avg (last 20):   11.6
Episode   80 | Return:   22.0 | Moving avg (last 20):   22.7
Episode  100 | Return:   41.0 | Moving avg (last 20):   91.2
Episode  120 | Return:   58.0 | Moving avg (last 20):   53.0
Episode  140 | Return:  216.0 | Moving avg (last 20):  173.1
Episode  160 | Return:   22.0 | Moving avg (last 20):  187.9
Episode  180 | Return:  153.0 | Moving avg (last 20):  328.4
Episode  200 | Return:  500.0 | Moving avg (last 20):  436.1

Environment solved in 219 episodes!
[GUI] Episode 1: total reward = 500.0, steps = 500
[GUI] Episode 2: total reward = 500.0, steps = 500
[GUI] Episode 3: total reward = 500.0, steps = 500
